In [1]:
import requests
import json
import pandas as pd
import numpy as np
from tqdm import tqdm

##나의 key
key = 'bf388499b71a365d725e1c888201736f7409d7e4'

In [2]:
#### HS Code 를 확인한다

path = r"/DATA/미국_500대_수출금액_.HScode_202508.xlsx"

hs_raw = pd.read_excel(path)

hs_raw['hs_code'] = hs_raw['HS_Code'].astype(str).str[:6]
# print(len(hs_code))

hs_code = hs_raw['hs_code'].unique().tolist()
# hs_code = hs_code[:3]
hs_code

['880000',
 '270900',
 '710812',
 '271019',
 '988000',
 '271111',
 '271012',
 '711590',
 '854231',
 '847330',
 '300490',
 '851762',
 '271112',
 '870323',
 '300215',
 '847150',
 '100590',
 '901890',
 '120190',
 '870899',
 '854239',
 '851713',
 '300212',
 '711319',
 '848620',
 '293719',
 '710239',
 '382219',
 '847180',
 '870324',
 '271121',
 '270112',
 '300214',
 '980110',
 '852351',
 '392690',
 '870431',
 '853710',
 '870840',
 '840734',
 '848180',
 '901839',
 '870829',
 '290110',
 '847170',
 '848690',
 '520100',
 '841199',
 '210690',
 '847130',
 '880730',
 '850440',
 '732690',
 '390140',
 '100199',
 '382499',
 '390120',
 '901819',
 '970191',
 '847149',
 '740400',
 '230400',
 '271113',
 '853890',
 '847989',
 '870380',
 '300241',
 '854370',
 '840820',
 '720449',
 '930690',
 '330499',
 '80212',
 '390110',
 '470321',
 '850760',
 '902139',
 '853690',
 '760200',
 '20130',
 '300439',
 '710391',
 '854442',
 '870340',
 '853669',
 '20230',
 '870333',
 '902190',
 '841112',
 '20329',
 '854430',
 '8

In [3]:
start_year = '2016-03-01'
start_y = 2025
start_q = 1
export_import = 'expDlr'

# 월말 날짜 생성
dates_period = pd.date_range(start='2020-01', end='2025-09', freq='ME')

dates_list1 = []
for dates in dates_period:
    temp = str(dates)[:7]
    dates_list1.append(temp)


In [4]:
def get_us_export_data(hs_list, start='2013-01', end='2025-09', api_key='your_key_here'):
    """
    미국 HS 코드별 수출 데이터를 월별로 가져옵니다.

    Parameters:
        hs_list (list): 조회할 HS 코드 리스트
        start (str): 시작 날짜 (yyyy-mm)
        end (str): 종료 날짜 (yyyy-mm)
        api_key (str): U.S. Census API 키

    Returns:
        pd.DataFrame: 월별 수출 데이터
        pd.DataFrame: 분기별 수출 데이터
    """
    us_export_hs = []
    date_range = pd.date_range(start=start, end=end, freq='MS')  # 매월 시작일

    total_steps = len(hs_list) * len(date_range)
    with tqdm(total=total_steps, desc="미국 수출 데이터 다운로드 중") as pbar:
        for hs in hs_list:
            for dt in date_range:
                year = dt.strftime('%Y')
                month = dt.strftime('%m')
                url = (
                    f"https://api.census.gov/data/timeseries/intltrade/exports/hs"
                    f"?get=ALL_VAL_MO&key={api_key}&YEAR={year}&MONTH={month}&E_COMMODITY={hs}"
                )
                try:
                    res = requests.get(url)
                    if res.status_code == 200:
                        data = json.loads(res.text)
                        if len(data) > 1:
                            temp = data[1]
                            us_export_hs.append(temp)
                    else:
                        print(f"❌ 실패: {year}-{month} {hs} → Status: {res.status_code}")
                except Exception as e:
                    print(f"⚠️ 예외 발생: {year}-{month} {hs} → {e}")
                pbar.update(1)

    if not us_export_hs:
        print("❌ 가져온 데이터가 없습니다.")
        return None, None

    # 데이터프레임 생성 및 컬럼 설정
    df = pd.DataFrame(us_export_hs, columns=['expDlr', 'year', 'month', 'hs_code'])

    df['expDlr'] = pd.to_numeric(df['expDlr'], errors='coerce')
    df.loc[df['expDlr'] > 1e18, 'expDlr'] = np.nan

    df['date'] = pd.to_datetime(df['year'] + '-' + df['month'], errors='coerce') + pd.offsets.MonthEnd(0)
    df.dropna(subset=['date'], inplace=True)
    df.set_index('date', inplace=True)
    df['quarter'] = df.index.to_period('Q')

    df_monthly = df.copy()
    df_quarterly = df.groupby(['quarter', 'hs_code'])['expDlr'].sum().reset_index()
    df_quarterly['quarter'] = df_quarterly['quarter'].dt.to_timestamp()

    return df_monthly, df_quarterly

def upload_trade_data_to_db(df, db_info, table_name='us_trade_data'):
    """
    미국 월별 수출 데이터를 지정한 DB 테이블에 업로드합니다.

    Parameters:
        df (DataFrame): 업로드할 월별 수출 데이터프레임
        db_info (dict): DB 접속 정보 (user, password, host, port, database)
        table_name (str): 업로드할 테이블 이름
    """
    from sqlalchemy import create_engine

    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )

    try:
        df_reset = df.reset_index()
        df_reset.to_sql(name=table_name, con=engine, if_exists='replace', index=False)
        print(f"✅ 데이터가 '{table_name}' 테이블에 성공적으로 업로드되었습니다.")
    except Exception as e:
        print(f"❌ DB 업로드 실패: {e}")


In [5]:
# hs_code = ['854231']
us_export_month, us_export_quarter = get_us_export_data(
    hs_list= hs_code,
    start='2020-04',
    end='2025-11',
    api_key= key  # 실제 API 키로 바꾸세요
)

미국 수출 데이터 다운로드 중:   0%|          | 40/34000 [00:37<8:50:07,  1.07it/s]


KeyboardInterrupt: 

In [6]:
us_export_month.tail(13)

,expDlr,year,month,hs_code,quarter
date,,,,,
2024-09-30,2.864393e+09,2024,09,854231,2024Q3
2024-10-31,2.994437e+09,2024,10,854231,2024Q4
2024-11-30,2.800374e+09,2024,11,854231,2024Q4
2024-12-31,2.817561e+09,2024,12,854231,2024Q4
2025-01-31,3.278119e+09,2025,01,854231,2025Q1
2025-02-28,2.762375e+09,2025,02,854231,2025Q1
2025-03-31,2.872100e+09,2025,03,854231,2025Q1
2025-04-30,3.127032e+09,2025,04,854231,2025Q2
2025-05-31,2.816068e+09,2025,05,854231,2025Q2


In [40]:
def get_us_import_data(hs_list, start='2013-01', end='2025-05', api_key='your_key_here',
                       value_var='GEN_VAL_MO', comm_lvl='HS6'):
    """
    미국 HS 코드별 '수입' 월별/분기 데이터
    - value_var: 'GEN_VAL_MO'(총수입) or 'CON_VAL_MO'(소비재 수입)
    - comm_lvl : 'HS2'/'HS4'/'HS6'/'HS10'
    """
    assert value_var in {'GEN_VAL_MO', 'CON_VAL_MO'}
    base = "https://api.census.gov/data/timeseries/intltrade/imports/hs"
    time_pred = f"from+{start}+to+{end}"

    parts = []
    with tqdm(total=len(hs_list), desc="미국 수입 데이터 다운로드 중") as pbar:
        for hs in hs_list:
            url = (f"{base}?get={value_var},I_COMMODITY"
                   f"&time={time_pred}&I_COMMODITY={hs}&COMM_LVL={comm_lvl}&key={api_key}")
            try:
                r = requests.get(url, timeout=30)
                if r.status_code == 200:
                    data = json.loads(r.text)
                    if len(data) > 1:
                        header, body = data[0], data[1:]
                        dfp = pd.DataFrame(body, columns=header)
                        dfp['__req_hs__'] = str(hs)  # 안전용(응답 검증용)
                        parts.append(dfp)
                else:
                    print(f"❌ 실패: {hs} → Status {r.status_code}")
            except Exception as e:
                print(f"⚠️ 예외: {hs} → {e}")
            pbar.update(1)

    if not parts:
        print("❌ 가져온 데이터가 없습니다.")
        return None, None

    df_raw = pd.concat(parts, ignore_index=True)

    # --- 날짜 컬럼 만들기: time(있으면 우선), 없으면 YEAR+MONTH ---
    if 'time' in df_raw.columns:
        df_raw['date'] = pd.to_datetime(df_raw['time'], format='%Y-%m', errors='coerce') + pd.offsets.MonthEnd(0)
        df_raw['year'] = df_raw['date'].dt.strftime('%Y')
        df_raw['month'] = df_raw['date'].dt.strftime('%m')
    else:
        df_raw['date'] = pd.to_datetime(df_raw['YEAR'].astype(str) + '-' + df_raw['MONTH'].astype(str),
                                        format='%Y-%m', errors='coerce') + pd.offsets.MonthEnd(0)
        df_raw['year'] = df_raw['YEAR'].astype(str)
        df_raw['month'] = df_raw['MONTH'].astype(str).str.zfill(2)

    # --- HS 코드 컬럼을 'Series'로 안전 추출 ---
    # 우선순위: I_COMMODITY -> I_COMMODITY_SDESC/Ldesc 등 'I_COMMODITY'로 시작하는 첫 열
    _candidates_exact = [c for c in df_raw.columns if c.upper() == 'I_COMMODITY']
    _candidates_prefix = [c for c in df_raw.columns if c.upper().startswith('I_COMMODITY')]
    if _candidates_exact:
        hs_src = _candidates_exact[0]
    elif _candidates_prefix:
        hs_src = _candidates_prefix[0]
    else:
        raise KeyError("I_COMMODITY* 열을 찾지 못했습니다. API 응답 헤더를 확인해 주세요.")

    hs_series = df_raw[hs_src]
    # 동일 이름 중복 등으로 DataFrame이 올 경우 첫 컬럼만 사용
    if isinstance(hs_series, pd.DataFrame):
        hs_series = hs_series.iloc[:, 0]

    df_raw['hs_code'] = hs_series.astype(str)

    # 자리수 패딩
    if comm_lvl == 'HS6':
        df_raw['hs_code'] = df_raw['hs_code'].str.zfill(6)
    elif comm_lvl == 'HS4':
        df_raw['hs_code'] = df_raw['hs_code'].str.zfill(4)
    elif comm_lvl == 'HS2':
        df_raw['hs_code'] = df_raw['hs_code'].str.zfill(2)

    # 금액/기타 정리
    df_raw[value_var] = pd.to_numeric(df_raw[value_var], errors='coerce')
    df_raw.loc[df_raw[value_var] > 1e18, value_var] = np.nan  # 이상치 가드

    # 최종 월별/분기별 구성
    df_raw = df_raw.dropna(subset=['date']).set_index('date').sort_index()
    df_raw.rename(columns={value_var: 'impDlr'}, inplace=True)
    df_raw['quarter'] = df_raw.index.to_period('Q')

    df_monthly = df_raw[['hs_code', 'impDlr', 'year', 'month']].copy()
    df_quarterly = (df_raw.groupby(['quarter', 'hs_code'])['impDlr'].sum().reset_index())
    df_quarterly['quarter'] = df_quarterly['quarter'].dt.to_timestamp()

    return df_monthly, df_quarterly

In [51]:
us_import_month, us_import_quarter = get_us_import_data(
    hs_list= hs_code,
    start='2013-04',
    end='2025-09',
    api_key= key  # 실제 API 키로 바꾸세요
)

미국 수입 데이터 다운로드 중:  81%|████████  | 403/500 [10:41<02:27,  1.52s/it]

❌ 실패: 40620 → Status 204


미국 수입 데이터 다운로드 중:  83%|████████▎ | 414/500 [10:57<01:57,  1.37s/it]

❌ 실패: 40610 → Status 204


미국 수입 데이터 다운로드 중:  85%|████████▌ | 426/500 [11:16<01:50,  1.50s/it]

...

미국 수입 데이터 다운로드 중:  88%|████████▊ | 438/500 [11:33<01:30,  1.47s/it]

❌ 실패: 20649 → Status 204


미국 수입 데이터 다운로드 중:  93%|█████████▎| 464/500 [12:12<00:46,  1.28s/it]

❌ 실패: 40410 → Status 204


미국 수입 데이터 다운로드 중: 100%|██████████| 500/500 [13:06<00:00,  1.57s/it]


In [54]:
db_info = {
    # 'host' : '192.168.0.230',
    'host' : 'hystox74.synology.me',
    'port' : 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database' : 'investar'
}

upload_trade_data_to_db(us_import_month, db_info, table_name='us_import_data')

✅ 데이터가 'us_import_data' 테이블에 성공적으로 업로드되었습니다.


In [55]:
# us_import_month

,hs_code,impDlr,year,month
date,,,,
2013-04-30,270900,2.302202e+10,2013,04
2013-04-30,330300,1.648706e+08,2013,04
2013-04-30,391740,2.936637e+07,2013,04
2013-04-30,843311,3.074331e+07,2013,04
2013-04-30,320910,7.503170e+06,2013,04
...,...,...,...,...
2025-06-30,220720,2.497407e+06,2025,06
2025-06-30,851779,1.690690e+08,2025,06
2025-06-30,271020,2.068000e+03,2025,06
